# Backpropagation — Hands-on Tutorial

In this notebook you will build intuition for:
1. How the forward pass constructs a computation graph step by step
2. How gradients flow backward through two layers — by hand, explicitly
3. How PyTorch's autograd computes the same gradients automatically
4. Why sigmoid activations cause vanishing gradients in deep networks

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| **→ 3** | **Backpropagation** | **`03_backprop_training.pdf`** | **`03_backpropagation.ipynb`** |
| 4 | Optimizers | `07_optimizers.pdf` *(new)* | `04_optimizers.ipynb` |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| 6 | Modern architectures | `05a_attention.pdf` + `05b_practical.pdf` | *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** You used `loss.backward()` in notebook 02 as a black box; now you'll see exactly what it computes.

**Leading to:** Notebook 04 takes the gradients you now understand and feeds them into tunable optimizers.

**If you skipped ahead:** You need the four-step training loop (forward, loss, backward, step) from notebook 02.


In [ ]:
# Run on Colab only — skip on JupyterHub (packages are pre-installed)
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets torch

import numpy as np
import torch
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, ToggleButtons

%matplotlib inline

# ── A single galaxy, two photometric colours ──────────────────────────────
# We work with one data point throughout Parts 1 and 2 so every number is visible.
# The task: classify transit (0) vs artifact (1) from two features.

torch.manual_seed(42)
np.random.seed(42)

x = torch.tensor([[1.1, 3.8]], dtype=torch.float32)   # dip depth, duration
y = torch.tensor([[0.0]], dtype=torch.float32)          # transit

print(f"Input x : {x.numpy()}")
print(f"Label y : {y.item()} (0 = transit, 1 = artifact)")
print()
print("Network: 2 → 4 → 1  (2 features → 4 hidden → 1 output)")

---

## The mathematics of backpropagation

### The chain rule

For a composed function $L = f(g(h(w)))$, the derivative with respect to $w$ is:

$$\frac{dL}{dw} = \frac{dL}{df} \cdot \frac{df}{dg} \cdot \frac{dg}{dh} \cdot \frac{dh}{dw}$$

Each factor is a local derivative — computed at the current value of its input. Backpropagation is simply the chain rule applied systematically from output to input.

For our 2 → 4 → 1 network with ReLU hidden and sigmoid output:

$$L \xrightarrow{\partial L/\partial \hat{y}} \hat{y} \xrightarrow{\partial \hat{y}/\partial z_2} z_2 \xrightarrow{\partial z_2/\partial W_2} W_2$$

| Factor | Expression | What it is |
|--------|-----------|------------|
| $\partial L / \partial \hat{y}$ | $-y/\hat{y} + (1-y)/(1-\hat{y})$ | BCE loss gradient |
| $\partial \hat{y} / \partial z_2$ | $\hat{y}(1-\hat{y})$ | Sigmoid derivative |
| Combined $\partial L / \partial z_2$ | $\hat{y} - y$ | **Clean cancellation** — the two terms simplify |
| $\partial z_2 / \partial W_2$ | $a_1^\top$ | Hidden activations as outer product |

### The ReLU derivative

$$\text{ReLU}(z) = \max(0, z), \qquad \frac{d}{dz}\text{ReLU}(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

Gradient either passes through unchanged (1) or is completely blocked (0). No shrinkage — this is what prevents vanishing gradients.

In [ ]:
# ── Chain rule step by step: compute each factor separately ──────────────────

torch.manual_seed(0)
W1_demo = torch.randn(2, 4)
b1_demo = torch.zeros(4)
W2_demo = torch.randn(4, 1)
b2_demo = torch.zeros(1)

x_demo = torch.tensor([[1.1, 3.8]])   # our single galaxy
y_demo = torch.tensor([[0.0]])         # transit label

# Forward pass
z1_d = x_demo @ W1_demo + b1_demo     # (1,4)  pre-activation layer 1
a1_d = torch.relu(z1_d)               # (1,4)  post-activation layer 1
z2_d = a1_d @ W2_demo + b2_demo       # (1,1)  pre-activation output
yhat = torch.sigmoid(z2_d)            # (1,1)  predicted probability

print(f"Forward pass:")
print(f"  z2    = {z2_d.item():.4f}")
print(f"  ŷ     = {yhat.item():.4f}   (probability of being artifact)")
print(f"  y     = {y_demo.item():.1f}   (true: transit)")
print()

# ── Chain rule factors ────────────────────────────────────────────────────────
dL_dyhat = -y_demo / yhat + (1 - y_demo) / (1 - yhat)
dyhat_dz2 = yhat * (1 - yhat)
dL_dz2_long = dL_dyhat * dyhat_dz2       # via chain rule
dL_dz2_short = yhat - y_demo             # clean cancellation

print(f"Chain rule factors at output layer:")
print(f"  ∂L/∂ŷ          = {dL_dyhat.item():.4f}")
print(f"  ∂ŷ/∂z₂  (σ′)  = {dyhat_dz2.item():.4f}")
print(f"  ∂L/∂z₂  (product) = {dL_dz2_long.item():.4f}")
print(f"  ŷ − y          = {dL_dz2_short.item():.4f}   ← same value: BCE+sigmoid cancels cleanly")
print()

# ── Visualise ReLU and its derivative ────────────────────────────────────────
z_plot = np.linspace(-3, 3, 300)
relu   = np.maximum(0, z_plot)
drelu  = (z_plot > 0).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z_plot, relu, lw=2.5, color='darkorange')
axes[0].axhline(0, color='gray', lw=0.7); axes[0].axvline(0, color='gray', lw=0.7)
axes[0].fill_between(z_plot, relu, where=(z_plot < 0), alpha=0.15, color='red',
                     label='Dead zone: output = 0')
axes[0].set_xlabel('z'); axes[0].set_ylabel('ReLU(z)')
axes[0].set_title('ReLU activation'); axes[0].legend()

axes[1].step(z_plot, drelu, lw=2.5, color='green', where='post')
axes[1].fill_between(z_plot, drelu, where=(z_plot < 0), alpha=0.15, color='red',
                     label='Blocked: gradient = 0')
axes[1].fill_between(z_plot, drelu, where=(z_plot > 0), alpha=0.15, color='green',
                     label='Pass-through: gradient = 1')
axes[1].set_xlabel('z'); axes[1].set_ylabel("d/dz ReLU(z)")
axes[1].set_title("ReLU derivative — binary gate\ngradient either passes or is blocked")
axes[1].legend()

plt.tight_layout(); plt.show()


### Think about it

- The clean cancellation $\partial L/\partial z_2 = \hat{y} - y$ is specific to   BCE loss paired with sigmoid output. What would the output gradient look like   for MSE loss with a sigmoid output — would it still be simple?
- ReLU's derivative is a binary gate: 1 or 0. What happens to the gradient flowing   through a neuron whose pre-activation $z$ is always negative during training?   Can the weights of that neuron ever change?
- In the chain rule product $\frac{dL}{dw} = \frac{dL}{dz_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w}$,   which factor is the ReLU derivative? What matrix shape does each factor have?
- If the network had 10 hidden layers all with sigmoid activations, and each   $\sigma'(z) \approx 0.2$ at the current weights, what is the approximate   magnitude of the gradient reaching layer 1?

---

## Part 1: The forward pass — step by step

We trace one data point through a 2 → 4 → 1 network, printing every intermediate tensor.
This is exactly the computation graph that PyTorch records during a real forward pass.

*The setup is binary classification with two features — what we'll learn applies to any small MLP.*

In [ ]:
# Initialise weights — requires_grad=True tells PyTorch to track these
torch.manual_seed(0)
W1 = torch.randn(2, 4, requires_grad=True)   # (n_in, n_hidden)
b1 = torch.zeros(4,    requires_grad=True)
W2 = torch.randn(4, 1, requires_grad=True)   # (n_hidden, n_out)
b2 = torch.zeros(1,    requires_grad=True)

# Layer 1: linear + ReLU
z1 = x @ W1 + b1                # (1,2) @ (2,4) = (1,4)
a1 = torch.relu(z1)

# Layer 2: linear + sigmoid
z2 = a1 @ W2 + b2               # (1,4) @ (4,1) = (1,1)
y_hat = torch.sigmoid(z2)

# Binary cross-entropy loss
loss = -(y * torch.log(y_hat + 1e-7) + (1 - y) * torch.log(1 - y_hat + 1e-7)).mean()

print(f"z1     : {z1.detach().numpy().flatten()}")
print(f"a1     : {a1.detach().numpy().flatten()}")
print(f"z2     : {z2.item():.6f}")
print(f"y_hat  : {y_hat.item():.6f}  (predicted P(artifact))")
print(f"loss   : {loss.item():.6f}")
print()
print(f"True label: {y.item():.0f}  — prediction {'✓' if (y_hat.item() < 0.5) == (y.item() == 0) else '✗'}")

### Think about it

- Look at `a1`. Some entries are 0 — those are dead ReLU units for this input. Which entries of `z1` caused them?
- The network outputs `y_hat ≈ ?`. Is it closer to 0 (transit) or 1 (artifact)? Is the prediction correct?
- What would `y_hat` be if all weights were zero? What would the loss be in that case?
- Why do we add `1e-7` inside `torch.log()`? What would happen without it?

---

## Part 2: The backward pass — by hand

We now compute the gradients manually, following the chain rule backward through the graph.
The derivations use the values printed in Part 1.

For binary cross-entropy with sigmoid output, the output gradient simplifies cleanly:

$$\frac{\partial L}{\partial z_2} = \hat{y} - y$$

From there, the chain rule gives the rest:

$$\frac{\partial L}{\partial W_2} = a_1^\top \cdot \frac{\partial L}{\partial z_2}, \qquad \frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} \cdot W_2^\top$$

$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \mathbf{1}[z_1 > 0], \qquad \frac{\partial L}{\partial W_1} = x^\top \cdot \frac{\partial L}{\partial z_1}$$

Run the cell below, then compare every gradient to Part 3 (autograd).

In [ ]:
# ── Manual backward pass ──────────────────────────────────────────────────
# Work with detached (no-gradient) tensors — we are doing the calculus ourselves.

y_hat_d = y_hat.detach()
a1_d    = a1.detach()
z1_d    = z1.detach()
W2_d    = W2.detach()

# Output layer gradient: d(BCE)/d(z2) = y_hat - y  (sigmoid + BCE cancel nicely)
dL_dz2 = y_hat_d - y                          # (1, 1)

# Gradients for layer 2 weights
dL_dW2 = a1_d.T @ dL_dz2                      # (4, 1)
dL_db2 = dL_dz2.sum(dim=0)                    # (1,)

# Gradient flowing back through W2 into a1
dL_da1 = dL_dz2 @ W2_d.T                      # (1, 4)

# ReLU gradient: 1 where z1 > 0, 0 elsewhere
dL_dz1 = dL_da1 * (z1_d > 0).float()          # (1, 4)

# Gradients for layer 1 weights
dL_dW1 = x.T @ dL_dz1                         # (2, 4)
dL_db1 = dL_dz1.sum(dim=0)                    # (4,)

print("Manual gradients:")
print(f"  dL/dW2 = {dL_dW2.numpy().flatten()}")
print(f"  dL/db2 = {dL_db2.numpy().flatten()}")
print(f"  dL/dW1 (first row) = {dL_dW1[0].numpy()}")
print(f"  dL/db1 = {dL_db1.numpy()}")

---

## Part 3: Autograd — PyTorch computes the same gradients

One call to `loss.backward()` populates `.grad` on every `requires_grad` tensor.
We verify it matches the manual computation above.

In [ ]:
loss.backward()

print("Autograd gradients:")
print(f"  W2.grad = {W2.grad.numpy().flatten()}")
print(f"  b2.grad = {b2.grad.numpy().flatten()}")
print(f"  W1.grad (first row) = {W1.grad[0].numpy()}")
print(f"  b1.grad = {b1.grad.numpy()}")
print()

# Verify agreement
tol = 1e-5
assert torch.allclose(W2.grad, dL_dW2, atol=tol), "W2 mismatch"
assert torch.allclose(b2.grad, dL_db2, atol=tol), "b2 mismatch"
assert torch.allclose(W1.grad, dL_dW1.T, atol=tol) or \
       torch.allclose(W1.grad.T, dL_dW1, atol=tol), "W1 mismatch"
assert torch.allclose(b1.grad, dL_db1, atol=tol), "b1 mismatch"
print("\u2713  Manual and autograd gradients agree to within 1e-5")

### Think about it

- The assertion on `W1.grad` checks two orientations. Why might the shapes differ between the manual and autograd versions? What convention does PyTorch use for weight matrix orientation?
- Change the activation from ReLU to sigmoid in the forward pass cell (Part 1) and re-run Parts 1–3. Do the manual gradient formulas still work? What changes?
- `loss.backward()` can only be called once on the same graph. Try calling it a second time — what error do you get? Why does PyTorch free the graph after the first backward pass?
- What would `W1.grad` look like if `a1` were all zeros? What does that imply for training?

---

## Part 4: Vanishing gradients

Each backward step multiplies the gradient by a local derivative.
For sigmoid, $\sigma'(z) \leq 0.25$ — always less than 1.
In a deep network, the gradient at an early layer is the product of many such factors.

The slider controls the number of sigmoid hidden layers.
Watch how the gradient norm at the first layer shrinks as depth increases.

In [ ]:
def grad_norms_by_depth(n_layers, activation='sigmoid'):
    """
    Build an n_layers-deep network, do one forward+backward pass,
    and return the gradient norm at each layer (from output to input).
    """
    torch.manual_seed(0)
    layers = []
    for _ in range(n_layers):
        layers += [nn.Linear(8, 8),
                   nn.Sigmoid() if activation == 'sigmoid' else nn.ReLU()]
    layers += [nn.Linear(8, 1), nn.Sigmoid()]
    model = nn.Sequential(*layers)

    x_in = torch.randn(1, 8)
    y_in = torch.tensor([[1.0]])
    out  = model(x_in)
    loss = nn.BCELoss()(out, y_in)
    loss.backward()

    norms = []
    for layer in model:
        if isinstance(layer, nn.Linear) and layer.weight.grad is not None:
            norms.append(layer.weight.grad.norm().item())
    return list(reversed(norms))   # index 0 = first (deepest) hidden layer

def plot_vanishing(n_layers, activation):
    norms_sig  = grad_norms_by_depth(n_layers, 'sigmoid')
    norms_relu = grad_norms_by_depth(n_layers, 'relu')

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.semilogy(range(1, len(norms_sig)+1),  norms_sig,
                'o-', color='tab:blue',  label='Sigmoid')
    ax.semilogy(range(1, len(norms_relu)+1), norms_relu,
                's-', color='tab:green', label='ReLU')
    ax.set_xlabel('Layer (1 = first hidden, rightmost = output)')
    ax.set_ylabel('Gradient norm (log scale)')
    ax.set_title(f'Gradient norms — {n_layers} hidden layers')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    ratio = norms_sig[0] / (norms_sig[-1] + 1e-12)
    print(f"Sigmoid: first-layer grad is {ratio:.2e}x the output-layer grad")

interact(
    plot_vanishing,
    n_layers=IntSlider(value=2, min=1, max=10, step=1, description='Depth'),
    activation=ToggleButtons(options=['sigmoid', 'relu'], value='sigmoid', description=''),
);

### Think about it

- At depth 5, how many orders of magnitude smaller is the sigmoid gradient at the first layer compared to the output layer?
- The ReLU gradient norms are not constant across layers — they vary. Why? What would make them exactly constant?
- Set depth to 1. Do sigmoid and ReLU behave similarly? At what depth does sigmoid first become clearly problematic?
- If the first-layer gradient is effectively zero, what happens to those weights during training? Does the network still converge?

---

## Part 5 — Gradient magnitude: layer by layer

The slider in Part 4 shows that gradients shrink with depth for sigmoid networks. Here we make the numbers concrete: a bar chart of the gradient norm at each layer, for **sigmoid** vs **ReLU**, at depths 2, 4, and 8.

Each bar represents $\|\nabla_{W_k} L\|$ — the Frobenius norm of the gradient for that layer's weight matrix. A bar near zero means the layer receives almost no learning signal.

In [ ]:
# ── Part 5: gradient norms per layer — sigmoid vs ReLU ───────────────────────

def gradient_norms(n_layers, activation='sigmoid', width=8, seed=0):
    """
    Returns list of gradient norms [layer_1, layer_2, ..., layer_n]
    ordered from first (closest to input) to last (closest to output).
    """
    torch.manual_seed(seed)
    layers = []
    for _ in range(n_layers):
        layers += [nn.Linear(width, width),
                   nn.Sigmoid() if activation == 'sigmoid' else nn.ReLU()]
    layers += [nn.Linear(width, 1), nn.Sigmoid()]
    model = nn.Sequential(*layers)

    x_in = torch.randn(1, width)
    y_in = torch.ones(1, 1)
    out  = model(x_in)
    loss = nn.BCELoss()(out, y_in)
    loss.backward()

    norms = []
    for layer in model:
        if isinstance(layer, nn.Linear) and layer.weight.grad is not None:
            norms.append(layer.weight.grad.norm().item())
    return norms   # index 0 = first layer (input end)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, n_layers in zip(axes, [2, 4, 8]):
    norms_sig = gradient_norms(n_layers, activation='sigmoid')
    norms_relu = gradient_norms(n_layers, activation='relu')

    x = np.arange(len(norms_sig))
    w = 0.35
    bars_s = ax.bar(x - w/2, norms_sig,  w, label='Sigmoid', color='steelblue',  alpha=0.85)
    bars_r = ax.bar(x + w/2, norms_relu, w, label='ReLU',    color='darkorange', alpha=0.85)

    ax.set_yscale('log')
    ax.set_xlabel('Layer index  (0 = closest to input)')
    ax.set_ylabel('Gradient norm  (log scale)')
    ax.set_title(f'{n_layers}-hidden-layer network')
    ax.set_xticks(x)
    ax.set_xticklabels([f'L{i}' for i in range(len(norms_sig))])
    ax.legend()
    ax.axhline(1e-4, color='red', linestyle='--', lw=0.8, alpha=0.6, label='vanishing threshold')

plt.suptitle('Gradient norms per layer: Sigmoid vs ReLU', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nSigmoid — ratio of first to last gradient norm:')
for n in [2, 4, 8]:
    g = gradient_norms(n, 'sigmoid')
    print(f'  {n} layers: {g[0]:.2e} / {g[-1]:.2e}  = ratio {g[0]/g[-1]:.1e}')


### Think about it

- At 8 hidden layers with sigmoid, how many orders of magnitude smaller is the   first-layer gradient compared to the last? Would that layer learn anything meaningful?
- For ReLU at 8 layers, does the gradient shrink from last to first? Is the   scaling approximately uniform across layers, or are there sharp drops?
- A "dead neuron" occurs when ReLU's input is always negative — gradient is exactly   zero for that neuron forever. At 8 layers with ReLU, do you see any bars drop   suspiciously close to zero? What would you do about it?
- Batch normalisation (used in modern networks) normalises activations between layers.   How would that affect the gradient magnitudes shown here?

---

## Exercise: implement `manual_backward`

Given the intermediate values from a forward pass, compute all four gradients manually.
Your results must match PyTorch's autograd to within $10^{-5}$.

The network is the same 2 → 4 → 1 architecture from Part 1, with ReLU hidden and sigmoid output.
Loss is binary cross-entropy.

In [ ]:
def manual_backward(x, y, W1, b1, W2, b2, z1, a1, z2, y_hat):
    """
    Manual backward pass for a 2→4→1 network with ReLU hidden, sigmoid output, BCE loss.

    Parameters (all torch.Tensor, detached)
    ----------
    x, y      : input and label, shapes (1,2) and (1,1)
    W1, b1    : first-layer weights (2,4) and bias (4,)
    W2, b2    : second-layer weights (4,1) and bias (1,)
    z1, a1    : pre- and post-activation of hidden layer, shapes (1,4)
    z2, y_hat : pre- and post-activation of output, shapes (1,1)

    Returns
    -------
    dW1, db1, dW2, db2 : gradient tensors matching the weight shapes
    """
    # Step 1: output gradient  dL/dz2 = y_hat - y
    # Step 2: dL/dW2 = a1^T @ dL/dz2        shape (4,1)
    #         dL/db2 = dL/dz2.sum(dim=0)     shape (1,)
    # Step 3: dL/da1 = dL/dz2 @ W2^T         shape (1,4)
    # Step 4: dL/dz1 = dL/da1 * (z1 > 0)     shape (1,4)  [ReLU gradient]
    # Step 5: dL/dW1 = x^T @ dL/dz1          shape (2,4)
    #         dL/db1 = dL/dz1.sum(dim=0)      shape (4,)
    raise NotImplementedError("Fill in manual_backward")

In [ ]:
# Fresh forward pass with gradient tracking
torch.manual_seed(0)
W1t = torch.randn(2, 4, requires_grad=True)
b1t = torch.zeros(4,    requires_grad=True)
W2t = torch.randn(4, 1, requires_grad=True)
b2t = torch.zeros(1,    requires_grad=True)

z1t    = x @ W1t + b1t
a1t    = torch.relu(z1t)
z2t    = a1t @ W2t + b2t
y_hatt = torch.sigmoid(z2t)
losst  = nn.BCELoss()(y_hatt, y)
losst.backward()

# Call your function with detached intermediates
dW1, db1, dW2, db2 = manual_backward(
    x.detach(), y.detach(),
    W1t.detach(), b1t.detach(), W2t.detach(), b2t.detach(),
    z1t.detach(), a1t.detach(), z2t.detach(), y_hatt.detach(),
)

tol = 1e-5
ok = (torch.allclose(dW2, W2t.grad, atol=tol) and
      torch.allclose(db2, b2t.grad, atol=tol) and
      torch.allclose(dW1, W1t.grad, atol=tol) and
      torch.allclose(db1, b1t.grad, atol=tol))

if ok:
    print("\u2713  All four gradients match PyTorch autograd")
else:
    for name, manual, auto in [("dW2", dW2, W2t.grad), ("db2", db2, b2t.grad),
                                ("dW1", dW1, W1t.grad), ("db1", db1, b1t.grad)]:
        match = "\u2713" if torch.allclose(manual, auto, atol=tol) else "\u2717"
        print(f"  {match} {name}: max diff = {(manual - auto).abs().max().item():.2e}")

---

## Going further

### Part A — Go deeper

The backprop derivation above assumed ReLU hidden activation and sigmoid output with BCE loss. The clean cancellation $\partial L / \partial z_2 = \hat{y} - y$ only holds for that specific combination.

**Challenge:** rederive the output gradient for **MSE loss** ($L = (\hat{y} - y)^2$) with **sigmoid output**. Write out $\partial L / \partial z_2$ symbolically, implement it, and verify your `manual_backward` function still passes the test when you swap `nn.BCELoss()` for `nn.MSELoss()` in the test cell.

*Suggested approach:* apply the chain rule in two steps: $\partial L / \partial z_2 = (\partial L / \partial \hat{y}) \cdot (\partial \hat{y} / \partial z_2)$. The first term is $2(\hat{y} - y)$; the second is $\sigma'(z_2) = \hat{y}(1 - \hat{y})$. Multiply them. How does this compare to the BCE result?

### Part B — Lead forward

The vanishing gradient problem was largely solved by:
1. ReLU activations (shown in Part 4)
2. Careful weight initialisation (Xavier/Glorot, Kaiming/He)
3. Residual connections (skip connections in ResNet)
4. Batch normalisation

Residual connections change the gradient flow entirely. Instead of $\partial L / \partial z_k$ passing through $\sigma'$ at every layer, a skip connection adds a direct path: $\partial L / \partial z_k = \partial L / \partial z_{k+1} + \text{(direct path)}$.

**Challenge:** sketch (on paper or in a markdown cell) the computation graph for a single residual block: $\mathbf{a}_\text{out} = \text{ReLU}(W\mathbf{a}_\text{in} + b) + \mathbf{a}_\text{in}$. Trace the gradient from $\partial L / \partial \mathbf{a}_\text{out}$ back to $\partial L / \partial \mathbf{a}_\text{in}$. What is the minimum gradient norm that can flow back through this block, regardless of how saturated the ReLU is?

---

### References

| | |
|---|---|
| **Video** | 3Blue1Brown — *Backpropagation calculus* [youtube.com/watch?v=tIeHLnjs5U8](https://www.youtube.com/watch?v=tIeHLnjs5U8) |
| **Primary** | Rumelhart, Hinton & Williams (1986) — *Learning representations by back-propagating errors.* Nature 323, 533–536 |
| **Primary** | He et al. (2015) — *Deep Residual Learning for Image Recognition.* arxiv [arxiv.org/abs/1512.03385](https://arxiv.org/abs/1512.03385) — the ResNet paper; Section 3 explains why skip connections fix vanishing gradients |
| **Blog** | Karpathy — *Yes you should understand backprop* [karpathy.medium.com/yes-you-should-understand-backprop-e2f06eab496b](https://karpathy.medium.com/yes-you-should-understand-backprop-e2f06eab496b) |

---

> **Try the exercise yourself first.** The solution is below — scroll past it if you haven't attempted the exercise.


---

## Solution — `manual_backward`

The backward pass for this network follows four steps, mirroring the forward pass in reverse. The key simplification: BCE loss with sigmoid output gives a clean cancellation — $\partial L / \partial z_2 = \hat{y} - y$ with no chain-rule fraction to compute.

In [ ]:
def manual_backward(x, y, W1, b1, W2, b2, z1, a1, z2, y_hat):
    # ── Step 1: output layer gradient ─────────────────────────────────────
    # BCE + sigmoid: dL/dz2 = y_hat - y  (the two terms cancel cleanly)
    dL_dz2 = y_hat - y                    # (1, 1)

    # ── Step 2: gradients for W2 and b2 ──────────────────────────────────
    dL_dW2 = a1.T @ dL_dz2               # (4, 1)  — outer product
    dL_db2 = dL_dz2.squeeze()            # (1,)

    # ── Step 3: backpropagate through W2 into hidden layer ────────────────
    dL_da1 = dL_dz2 @ W2.T              # (1, 4)

    # ── Step 4: backpropagate through ReLU ────────────────────────────────
    # ReLU derivative is 1 where pre-activation z1 > 0, else 0
    dL_dz1 = dL_da1 * (z1 > 0).float()  # (1, 4)

    # ── Step 5: gradients for W1 and b1 ──────────────────────────────────
    dL_dW1 = x.T @ dL_dz1              # (2, 4)
    dL_db1 = dL_dz1.squeeze()           # (4,)

    return dL_dW1, dL_db1, dL_dW2, dL_db2

# ── What this connects back to ────────────────────────────────────────────
# Part 5 bar chart shows that sigmoid's small derivative (max 0.25) stacks up
# multiplicatively as you go deeper — that's exactly what dL_da1 * sigmoid'(z1)
# computes at each layer.  ReLU's derivative is either 0 or 1, so there is no
# systematic shrinkage — which is what the ReLU bars show.
